In [1]:
import pandas as pd
import ast

df = pd.read_csv('./ptbxl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/ptbxl_database.csv')

# load the scp_codes reference table
scp_df = pd.read_csv('./ptbxl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/scp_statements.csv', index_col=0)

# parse scp_codes from string to dict
df['scp_codes'] = df['scp_codes'].apply(ast.literal_eval)

# map to superclass using the reference table
def get_superclass(scp_dict):
    superclasses = []
    for code in scp_dict.keys():
        if code in scp_df.index:
            sc = scp_df.loc[code, 'diagnostic_class']
            if pd.notna(sc):
                superclasses.append(sc)
    return list(set(superclasses)) if superclasses else ['UNKNOWN']

df['superclass'] = df['scp_codes'].apply(get_superclass)

# explode to count
from collections import Counter
all_labels = [label for sublist in df['superclass'] for label in sublist]
print(Counter(all_labels))

Counter({'NORM': 9514, 'MI': 5469, 'STTC': 5235, 'CD': 4898, 'HYP': 2649, 'UNKNOWN': 411})


In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# add label back to df for plotting
label_names = {0: 'Normal', 1: 'Abnormal', 2: 'Critical(MI)'}
df['label_name'] = df['label'].map(label_names)

# columns worth plotting
plot_cols = ['age', 'sex', 'height', 'weight', 'heart_axis', 
             'baseline_drift', 'static_noise', 'burst_noise', 
             'electrodes_problems', 'extra_beats', 'pacemaker']

colors = ['green', 'orange', 'red']
classes = ['Normal', 'Abnormal', 'Critical(MI)']

fig, axes = plt.subplots(4, 3, figsize=(18, 20))
axes = axes.flatten()

for idx, col in enumerate(plot_cols):
    ax = axes[idx]
    
    if df[col].nunique() <= 10:  # categorical
        for i, cls in enumerate(classes):
            subset = df[df['label_name'] == cls][col].value_counts().sort_index()
            ax.bar([str(x) + f'_{cls[:3]}' for x in subset.index], 
                   subset.values, color=colors[i], alpha=0.7, label=cls)
        ax.set_title(f'{col} Distribution')
        ax.legend(fontsize=7)
        ax.tick_params(axis='x', rotation=45)
    
    else:  # continuous
        for i, cls in enumerate(classes):
            subset = df[df['label_name'] == cls][col].dropna()
            ax.hist(subset, bins=30, color=colors[i], 
                    alpha=0.6, label=cls, density=True)
        ax.set_title(f'{col} Distribution')
        ax.legend(fontsize=7)
    
    ax.set_ylabel('Count / Density')

# hide empty subplot
axes[-1].set_visible(False)

plt.suptitle('PTB-XL Dataset Feature Distribution by Class', 
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

KeyError: 'label'

In [10]:
def map_label(superclasses):
    if 'MI' in superclasses:
        return 2  # Critical
    elif 'NORM' in superclasses:
        return 0  # Normal
    else:
        return 1  # Abnormal

In [15]:
import pandas as pd
import numpy as np
import ast
import wfdb
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
import tensorflow as tf
from tensorflow.keras import layers, models

# ── 1. Load and label ──────────────────────────────────────────────
PATH = './ptbxl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/'

df = pd.read_csv(PATH + 'ptbxl_database.csv')
scp_df = pd.read_csv(PATH + 'scp_statements.csv', index_col=0)
df['scp_codes'] = df['scp_codes'].apply(ast.literal_eval)

def get_superclass(scp_dict):
    superclasses = []
    for code in scp_dict.keys():
        if code in scp_df.index:
            sc = scp_df.loc[code, 'diagnostic_class']
            if pd.notna(sc):
                superclasses.append(sc)
    return list(set(superclasses)) if superclasses else ['UNKNOWN']

df['superclass'] = df['scp_codes'].apply(get_superclass)

def map_label(superclasses):
    if 'MI' in superclasses:
        return 2  # Critical
    elif 'NORM' in superclasses:
        return 0  # Normal
    elif 'UNKNOWN' in superclasses:
        return -1  # drop these
    else:
        return 1  # Abnormal

df['label'] = df['superclass'].apply(map_label)
df = df[df['label'] != -1].reset_index(drop=True)
print("Label distribution:\n", df['label'].value_counts())

# ── 2. Load ECG signals (low-res 100Hz is enough) ─────────────────
def load_ecg(filename):
    record = wfdb.rdsamp(PATH + filename)
    return record[0]  # shape: (1000, 12)

print("Loading ECG signals... this takes a few minutes")
X = np.array([load_ecg(f) for f in df['filename_lr']])
y = df['label'].values
print("X shape:", X.shape)  # should be (N, 1000, 12)

# ── 3. Split ───────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── 4. Class weights ───────────────────────────────────────────────
classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, weights))
print("Class weights:", class_weight_dict)

# ── 5. CNN model ───────────────────────────────────────────────────
def build_cnn(input_shape=(1000, 12)):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv1D(32, kernel_size=5, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(2),
        layers.Conv1D(64, kernel_size=5, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(2),
        layers.Conv1D(128, kernel_size=3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling1D(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(3, activation='softmax')
    ])
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_cnn()
model.summary()

# ── 6. Train ───────────────────────────────────────────────────────
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    class_weight=class_weight_dict
)

# ── 7. Evaluate — accuracy is not enough, check per-class F1 ──────
y_pred = np.argmax(model.predict(X_test), axis=1)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, 
      target_names=['Normal', 'Abnormal', 'Critical(MI)']))

Label distribution:
 label
0    9513
1    6406
2    5469
Name: count, dtype: int64
Loading ECG signals... this takes a few minutes
X shape: (21388, 1000, 12)
Class weights: {np.int64(0): np.float64(0.7494524748138415), np.int64(1): np.float64(1.1128455284552845), np.int64(2): np.float64(1.3036190476190477)}


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 1000, 32)       │         1,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1000, 32)       │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 500, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 500, 64)        │        10,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 500, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 250, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 250, 128)       │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 250, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 46,307 (180.89 KB)

 Trainable params: 45,859 (179.14 KB)

 Non-trainable params: 448 (1.75 KB)

Epoch 1/20
482/482 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - accuracy: 0.6754 - loss: 0.7627 - val_accuracy: 0.6201 - val_loss: 0.8610
Epoch 2/20
482/482 ━━━━━━━━━━━━━━━━━━━━ 9s 19ms/step - accuracy: 0.7348 - loss: 0.6535 - val_accuracy: 0.7294 - val_loss: 0.6400
Epoch 3/20
482/482 ━━━━━━━━━━━━━━━━━━━━ 9s 18ms/step - accuracy: 0.7569 - loss: 0.6095 - val_accuracy: 0.7393 - val_loss: 0.6053
Epoch 4/20
482/482 ━━━━━━━━━━━━━━━━━━━━ 9s 19ms/step - accuracy: 0.7709 - loss: 0.5785 - val_accuracy: 0.7510 - val_loss: 0.5875
Epoch 5/20
482/482 ━━━━━━━━━━━━━━━━━━━━ 9s 19ms/step - accuracy: 0.7807 - loss: 0.5549 - val_accuracy: 0.7709 - val_loss: 0.5563
Epoch 6/20
482/482 ━━━━━━━━━━━━━━━━━━━━ 9s 18ms/step - accuracy: 0.7862 - loss: 0.5420 - val_accuracy: 0.7662 - val_loss: 0.5530
Epoch 7/20
482/482 ━━━━━━━━━━━━━━━━━━━━ 10s 21ms/step - accuracy: 0.7954 - loss: 0.5227 - val_accuracy: 0.7610 - val_loss: 0.5600
Epoch 8/20
482/482 ━━━━━━━━━━━━━━━━━━━━ 12s 25ms/step - accuracy: 0.8010 - loss: 0.5128 - val_a